# 04 - Exploratory Data Analysis

This notebook explores the cleaned analytical tables to surface patterns, anomalies and hypotheses before the formal business analysis.

All charts use Matplotlib + Seaborn; the dashboard in the next layer uses Plotly for interactivity.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

FINAL = ROOT / 'data' / 'final'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 110

fact = pd.read_csv(FINAL / 'fact_sales.csv', parse_dates=['order_date'])
monthly = pd.read_csv(FINAL / 'monthly_summary.csv')
by_cat = pd.read_csv(FINAL / 'category_summary.csv')
by_reg = pd.read_csv(FINAL / 'region_summary.csv')
by_ch  = pd.read_csv(FINAL / 'channel_summary.csv')
top20  = pd.read_csv(FINAL / 'top_products.csv')
rfm    = pd.read_csv(FINAL / 'rfm_segments.csv')

completed = fact[fact['is_completed']]
print(f'Completed transactions: {len(completed):,}')

## 1. Revenue & profit trend over time

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=monthly, x='year_month', y='revenue', ax=ax[0], marker='o', color='#1f77b4')
ax[0].set_title('Monthly Revenue')
ax[0].set_xlabel('Month'); ax[0].set_ylabel('Revenue (USD)')
ax[0].tick_params(axis='x', rotation=45)

sns.lineplot(data=monthly, x='year_month', y='profit', ax=ax[1], marker='o', color='#2ca02c')
ax[1].set_title('Monthly Profit')
ax[1].set_xlabel('Month'); ax[1].set_ylabel('Profit (USD)')
ax[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(REPORTS / 'monthly_revenue_profit.png', bbox_inches='tight')
plt.show()

## 2. Seasonality - month-of-year pattern

In [ ]:
completed = completed.copy()
completed['month'] = completed['order_date'].dt.month
season = completed.groupby('month').agg(revenue=('net_revenue', 'sum'), orders=('order_id', 'nunique'))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(x=season.index, y='revenue', data=season.reset_index(), ax=ax[0], color='#1f77b4')
ax[0].set_title('Revenue by Month-of-Year')
ax[0].set_xlabel('Month'); ax[0].set_ylabel('Revenue (USD)')

sns.barplot(x=season.index, y='orders', data=season.reset_index(), ax=ax[1], color='#ff7f0e')
ax[1].set_title('Orders by Month-of-Year')
ax[1].set_xlabel('Month'); ax[1].set_ylabel('Orders')

plt.tight_layout()
plt.savefig(REPORTS / 'seasonality.png', bbox_inches='tight')
plt.show()

## 3. Sales by category

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=by_cat.sort_values('revenue', ascending=False), y='category', x='revenue', ax=ax[0], color='#1f77b4')
ax[0].set_title('Revenue by Category'); ax[0].set_xlabel('Revenue (USD)')

sns.barplot(data=by_cat.sort_values('profit', ascending=False), y='category', x='profit', ax=ax[1], color='#2ca02c')
ax[1].set_title('Profit by Category'); ax[1].set_xlabel('Profit (USD)')

plt.tight_layout()
plt.savefig(REPORTS / 'category_revenue_profit.png', bbox_inches='tight')
plt.show()

## 4. Regional performance

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=by_reg.sort_values('revenue', ascending=False), x='region', y='revenue', ax=ax[0], palette='Blues_d')
ax[0].set_title('Revenue by Region'); ax[0].set_ylabel('Revenue (USD)')
ax[0].tick_params(axis='x', rotation=15)

sns.barplot(data=by_reg.sort_values('orders', ascending=False), x='region', y='orders', ax=ax[1], palette='Oranges_d')
ax[1].set_title('Orders by Region'); ax[1].set_ylabel('Orders')
ax[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(REPORTS / 'region_performance.png', bbox_inches='tight')
plt.show()

## 5. Order & price distributions

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(completed['net_revenue'], bins=60, ax=ax[0], color='#1f77b4')
ax[0].set_title('Line-item Revenue')
ax[0].set_xlabel('Revenue (USD)')
ax[0].set_xlim(0, completed['net_revenue'].quantile(0.99))

sns.histplot(completed['quantity'], bins=20, ax=ax[1], color='#2ca02c')
ax[1].set_title('Quantity per Line')
ax[1].set_xlabel('Quantity')

sns.histplot(completed['discount'], bins=20, ax=ax[2], color='#ff7f0e')
ax[2].set_title('Discount Distribution')
ax[2].set_xlabel('Discount')

plt.tight_layout()
plt.savefig(REPORTS / 'distributions.png', bbox_inches='tight')
plt.show()

## 6. Returns analysis

In [ ]:
ret = completed.groupby('category').agg(items=('returned_flag', 'size'), returned=('returned_flag', 'sum'))
ret['return_rate'] = ret['returned'] / ret['items']
ret = ret.sort_values('return_rate', ascending=False).reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=ret, x='category', y='return_rate', palette='Reds_d', ax=ax)
ax.set_title('Return Rate by Category')
ax.set_ylabel('Return rate')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.savefig(REPORTS / 'return_rate.png', bbox_inches='tight')
plt.show()

## 7. Correlation analysis

Pearson correlations between continuous variables.

> **Limitation:** correlation is **not causation**. For example, `discount` may correlate with `quantity` because discounts cause higher volumes *and* because high-volume products are intentionally discounted. Any business decision must combine correlation with domain context, A/B testing or causal inference.

In [ ]:
corr_cols = ['quantity', 'discount', 'unit_price_sold', 'unit_cost', 'net_revenue', 'profit']
corr = completed[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, square=True)
ax.set_title('Correlation matrix - line item metrics')
plt.tight_layout()
plt.savefig(REPORTS / 'correlation_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sample = completed.sample(min(5000, len(completed)), random_state=42)
sns.scatterplot(data=sample, x='unit_price_sold', y='quantity', alpha=0.3, ax=ax[0])
ax[0].set_title('Price vs Quantity sold'); ax[0].set_xlim(0, sample['unit_price_sold'].quantile(0.99))

sns.scatterplot(data=sample, x='discount', y='quantity', alpha=0.3, ax=ax[1])
ax[1].set_title('Discount vs Quantity sold')
plt.tight_layout()
plt.savefig(REPORTS / 'price_discount_vs_quantity.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=rfm, x='frequency', y='revenue', alpha=0.4, ax=ax, color='#9467bd')
ax.set_title('Customer frequency vs lifetime revenue')
ax.set_xlabel('Orders per customer'); ax.set_ylabel('Lifetime revenue (USD)')
ax.set_xlim(0, rfm['frequency'].quantile(0.99))
ax.set_ylim(0, rfm['revenue'].quantile(0.99))
plt.tight_layout()
plt.savefig(REPORTS / 'frequency_vs_value.png', bbox_inches='tight')
plt.show()

## 8. RFM segments

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
seg = rfm['segment'].value_counts().reset_index()
seg.columns = ['segment', 'customers']
sns.barplot(data=seg, x='customers', y='segment', ax=ax[0], palette='viridis')
ax[0].set_title('Customers per Segment')

rev = rfm.groupby('segment')['revenue'].sum().sort_values().reset_index()
sns.barplot(data=rev, x='revenue', y='segment', ax=ax[1], palette='magma')
ax[1].set_title('Total Revenue by Segment')
plt.tight_layout()
plt.savefig(REPORTS / 'rfm_segments.png', bbox_inches='tight')
plt.show()

## 9. Channel performance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=by_ch.sort_values('revenue', ascending=False), x='channel', y='revenue', palette='Set2', ax=ax)
ax.set_title('Revenue by Channel'); ax.set_ylabel('Revenue (USD)')
plt.tight_layout()
plt.savefig(REPORTS / 'channel_performance.png', bbox_inches='tight')
plt.show()

## 10. Top vs slow-moving products

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=top20.head(10), y='product_name', x='revenue', ax=ax[0], palette='Greens_r')
ax[0].set_title('Top 10 Products by Revenue'); ax[0].set_xlabel('Revenue (USD)')

slow = pd.read_csv(FINAL / 'top_products.csv')  # placeholder
from src.transformation import slow_moving_products
slow = slow_moving_products(fact, min_units=5).head(10)
sns.barplot(data=slow, y='product_name', x='units', ax=ax[1], palette='Reds_r')
ax[1].set_title('Slow-moving Products (<=5 units sold)'); ax[1].set_xlabel('Units sold')
plt.tight_layout()
plt.savefig(REPORTS / 'top_vs_slow_products.png', bbox_inches='tight')
plt.show()

## EDA takeaways

The next notebook converts these patterns into formal business findings and recommendations.